# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant schema URL:**
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}\nIdentifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Below, we list the `@id` of each record set as well as the `@id`s of the available fields/columns for each record set.

In [ ]:
# Explore available record sets and fields
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets defined in the top-level Croissant. Listing all detected via dataset.record_sets:")
    record_sets = [r for r in dataset.record_sets]

record_set_ids = []
for rs in dataset.record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print("  Fields / Columns:")
    for field in rs.fields:
        print(f"    - Name: {getattr(field, 'name', '')}, @id: {field['@id']}, dataType: {getattr(field, 'data_type', '?')}")
    print()

print(f"Available record set @ids: {record_set_ids}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s listed above.

In [ ]:
# The following code will extract the data from each record set, using their `@id`.
record_sets_to_extract = record_set_ids.copy()
dataframes = {}

for rs_id in record_sets_to_extract:
    print(f"Loading records for record set with @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()

# Choose the main record set for further EDA, here we select the first one for demonstration
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"First 5 rows from main record set: {main_rs_id}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's do some quick EDA: filter records based on a numeric field, normalize values, and group by a key.

Replace the `<field_id>` values below with the appropriate field `@id` from the overview above.

In [ ]:
# Example: filter and normalize using a numeric field, and group by a category
# Please update numeric_field_id and group_field_id below as appropriate for your dataset

# For demonstration, automatically select a numeric column if possible
import numpy as np

df = dataframes[main_rs_id].copy()
# Try to pick a numeric column
numeric_field_id = None
for col in df.columns:
    # Try to infer if numeric
    try:
        if np.issubdtype(df[col].dropna().astype(float).dtype, np.number):
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    print("Could not auto-detect a numeric field. Please update 'numeric_field_id'.")
else:
    print(f"Using numeric field for analysis: {numeric_field_id}")
    # Filter by threshold (here, use 10 or median value)
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
    except Exception:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    group_field_id = None
    for col in df.columns:
        # Heuristic: not numeric, not all unique
        if col == numeric_field_id:
            continue
        if df[col].dtype == object and df[col].nunique() < 0.5*len(df):
            group_field_id = col
            break

    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a group field was found, show group means
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.barplot(
            data=grouped_df,
            x=group_field_id,
            y=numeric_field_id
        )
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`, reviewed available record sets and fields, extracted tabular data, and performed basic exploratory analysis. For deeper insights, explore additional record sets, try field-specific filtering, or apply domain-specific statistical tests or visualizations.

**Key Takeaways:**
- The dataset contains rich protein-level data, and fields/columns are accessible via their `@id`s.
- `mlcroissant` provides a standards-compliant way to programmatically load and analyze FAIR datasets.
- For custom analyses, explore the record set and field descriptions and tailor transformations to your hypotheses.